# Session 4: From Prototype to Production — Sentiment, Triggers, and Portfolio Operations

We have a validated strategy: the Cobb-Douglas rebalancing engine passed Session 3's regime-shifting stress tests. We have a powerful adaptive tool: the epsilon-greedy bandit that learns which assets to include. Now we combine them into a **production system** — a daily operating loop where the bandit selects assets, the Cobb-Douglas allocator decides how much of each, sentiment monitoring adjusts risk appetite in real time, and human override rules provide the final safety net.

> __Learning Objectives:__
>
> * **Production Portfolio Loop**: Design a daily operating loop that combines bandit asset selection with Cobb-Douglas utility allocation, including sentiment-driven lambda overrides when conditions deteriorate
> * **Escalation Triggers and Human Override**: Define three escalation triggers (drawdown, sentiment crash, bandit churn) with severity levels and recommended actions that provide investment committee transparency
> * **Operational Dashboard and Stress Testing**: Build a dashboard that aggregates production metrics across simulations and stress-test the system with crash injection to validate the safety net

Let's take this from prototype to production.

## Examples
The following example notebooks accompany this lecture:

\u27a1 [Example: Production Portfolio Engine](eCornell-AI-Finance-L4-Example-ProductionPortfolioEngine-May-2026.ipynb). In this example, we build the full production loop — bandit + Cobb-Douglas working together — add sentiment monitoring with lambda overrides, and implement human escalation rules.

\u27a1 [Example: Operational Dashboard and Stress Report](eCornell-AI-Finance-L4-Example-OperationalDashboard-May-2026.ipynb). In this example, we build a daily operational dashboard, run production simulations across multiple HMM paths, and inject a crash scenario to test whether the system de-risks and escalates appropriately.

## From Backtest to Production: What Changes?

In Sessions 2-3, we built and tested the strategy in a **backtest** environment: all prices are known in advance, the engine runs over a fixed historical (synthetic) path, and there are no operational concerns. Production is different:

> **1. The asset universe drifts.** New assets become relevant; old ones become illiquid or delisted. The bandit must continuously learn which assets belong in the portfolio — not just once at initialization.

> **2. Parameters go stale.** SIM parameters ($\alpha_i$, $\beta_i$, $\sigma_i$) estimated from historical data decay as market structure changes. The sentiment signal may lose predictive power. The system needs monitoring to detect when its models are degrading.

> **3. Execution risk matters.** In backtesting, trades happen at the quoted price. In production, there's slippage, market impact, and latency. The turnover cap becomes a hard operational constraint, not just a simulation parameter.

> **4. Someone is watching.** An autonomous trading system needs transparency. Investment committees, compliance officers, and regulators need to understand _why_ the system made each decision. This requires logging, dashboards, and clear escalation protocols.

## The Production Loop: Bandit + Cobb-Douglas

In Sessions 2-3, the bandit and Cobb-Douglas engine competed against each other. In production, they work **together**:

> **Daily Production Loop:**
> 1. **Read sentiment.** Ingest the latest sentiment score (from news, filings, or synthetic generation). If sentiment is below the crisis threshold, override $\lambda$ to increase risk-aversion.
> 2. **Run bandit.** The epsilon-greedy bandit evaluates asset subsets using current prices, SIM parameters, and the effective $\lambda$. It selects the best asset subset.
> 3. **Check escalation triggers.** Before acting on the bandit's recommendation, check: Is the drawdown too deep? Did sentiment crash? Did the bandit change too many assets at once? If any trigger fires, log an escalation event.
> 4. **Execute allocation.** If no critical escalation: use the Cobb-Douglas allocator to compute optimal shares for the bandit-selected subset. Apply the turnover cap. Execute trades.
> 5. **If critical escalation:** De-risk to cash. Notify the investment committee.
> 6. **Log everything.** Record the day's state: shares, cash, wealth, bandit action, sentiment, lambda, whether we rebalanced, whether we escalated.

This loop runs once per trading day. The bandit adapts the asset subset over time; the Cobb-Douglas allocator adapts the allocation within that subset based on sentiment. The escalation triggers provide the human safety net.

## Sentiment Monitoring

In Sessions 2-3, the sentiment signal $\lambda_t$ came entirely from EMA crossover — a technical indicator derived from price. In production, we augment this with **external sentiment**: AI-extracted scores from news articles, earnings transcripts, and regulatory filings.

For this course, we generate synthetic sentiment scores correlated with market regime:

> **Synthetic Sentiment.** The sentiment score $s_t \in [-1, 1]$ is derived from short-term market returns plus noise, smoothed and mapped through $\tanh$:
>
> $$s_t = \tanh\left(10 \cdot r_{t,5d} + \sigma_{\text{noise}} \cdot z_t\right)$$
>
> where $r_{t,5d}$ is the 5-day return and $z_t$ is Gaussian noise.

When sentiment drops below a threshold (e.g., $s_t < -0.5$), the system overrides $\lambda$ to a high risk-aversion value. This is a **defensive circuit breaker** — it doesn't wait for the drawdown to materialize before pulling back.

| Sentiment Level | Action |
|----------------|--------|
| $s_t > 0$ | Normal operations, use EMA-based $\lambda$ |
| $-0.5 \leq s_t \leq 0$ | Caution zone, log warning |
| $s_t < -0.5$ | Override $\lambda$ to defensive value, escalate |

## Escalation Triggers and Human Override

The production system defines three escalation triggers:

| Trigger | Condition | Severity | Action |
|---------|-----------|----------|--------|
| **Drawdown** | Portfolio down > $d_{\max}$ from peak | Critical | De-risk to 100% cash. Notify investment committee immediately. |
| **Sentiment Crash** | Sentiment $s_t <$ threshold | Warning | Override $\lambda$ to defensive. Log for review. Continue operating. |
| **Bandit Churn** | Bandit changes > $N$ assets in one day | Warning | Hold previous allocation (ignore bandit). Log for review. |

> **Critical vs. Warning.** A **critical** escalation halts trading and moves to cash — this is the nuclear option. A **warning** modifies behavior (override lambda, ignore bandit) but keeps the system running. Both are logged for post-hoc review.

> **Investment Committee Transparency.** Every escalation event includes:
> - When it happened (day)
> - What triggered it (drawdown/sentiment/churn)
> - How severe it is (warning/critical)
> - What the system recommends
>
> This creates an audit trail that the investment committee can review. The system is autonomous but _accountable_ — every decision can be traced and justified.

## Summary

> __Key Takeaways:__
>
> * **Production differs fundamentally from backtesting**: Asset universes drift, parameters go stale, execution has friction, and humans need transparency — the bandit + Cobb-Douglas production loop addresses these by adapting daily with sentiment-driven lambda overrides
> * **Escalation triggers provide the human safety net**: Drawdown breaches trigger critical de-risking to cash; sentiment crashes and bandit churn trigger warnings that modify behavior while keeping the system running — every event is logged for investment committee review
> * **The four-session arc is complete**: From classical portfolio construction (Session 1), through utility-based AI allocation (Session 2), regime-switching stress testing (Session 3), to production deployment with operational safeguards — each session builds on the last

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The production system described is a pedagogical tool using synthetic data. Real-world deployment requires regulatory approval, compliance review, operational infrastructure, and risk management beyond the scope of this course.